# Lights Out and Away — F1 ML Project

## Predicting F1 Race Winners and Pit Stop Durations using Supervised Machine Learning

**Course:** Fundamentals of AI/ML  
**Algorithms:** Linear Regression & K-Nearest Neighbors  
**Data Source:** FastF1 Python Library (2022, 2023, 2024 Seasons)  

---

### Notebook Overview
1. **Setup** — Import libraries and configure the F1 theme
2. **Data Collection** — Load F1 data from CSV files
3. **Data Preprocessing** — Load processed, encoded features
4. **Exploratory Data Analysis** — Understand patterns through plots
5. **Problem 1: Race Winner Prediction** — Train LR and KNN, evaluate metrics
6. **Problem 2: Pit Stop Duration Prediction** — Train LR and KNN regressor
7. **Model Comparison** — Compare all four models side by side
8. **Conclusion** — Key findings and future work


---
## 1. Setup

We import all required libraries and apply the F1 dark-navy theme to every visualization in this notebook.

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print('All libraries imported!')

In [ ]:
# F1 Visual Theme
F1_STYLE = {
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e8002d',   'axes.labelcolor': '#ffffff',
    'axes.titlecolor': '#ffffff',  'xtick.color': '#ffffff',
    'ytick.color': '#ffffff',      'grid.color': '#2a2a4a',
    'grid.alpha': 0.5,             'text.color': '#ffffff',
    'legend.facecolor': '#1a1a2e', 'legend.edgecolor': '#e8002d',
}
TEAM_COLORS = {
    'Red Bull Racing':'#3671C6','Ferrari':'#E8002D','Mercedes':'#27F4D2',
    'McLaren':'#FF8000','Aston Martin':'#229971','Alpine':'#FF87BC',
    'Williams':'#64C4FF','RB':'#6692FF','Kick Sauber':'#52E252','Haas F1 Team':'#B6BABD'
}
TYRE_COLORS = {'SOFT':'#E8002D','MEDIUM':'#FFF200','HARD':'#FFFFFF','INTERMEDIATE':'#39B54A','WET':'#0067FF'}
F1_RED = '#e8002d'; F1_BG = '#1a1a2e'; F1_ORANGE = '#ff7c43'
plt.rcParams.update(F1_STYLE)
print('F1 theme applied!')

---
## 2. Data Collection

We use FastF1 to collect real F1 data from 2022–2024. The data is saved to `data/raw/` the first time `main.py` runs. If raw CSVs already exist, we load them directly.

Data collected includes: race results, qualifying positions, pit stop durations, weather conditions, circuit info, and tyre stints.

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

def load_csv(filename, directory=RAW_DIR):
    path = os.path.join(directory, filename)
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

race_raw    = load_csv('race_data_raw.csv')
pitstop_raw = load_csv('pitstop_data_raw.csv')
weather_raw = load_csv('weather_data_raw.csv')
tyre_raw    = load_csv('tyre_data_raw.csv')

print(f'Race data:     {race_raw.shape}')
print(f'Pit stop data: {pitstop_raw.shape}')
print(f'Weather data:  {weather_raw.shape}')
print(f'Tyre data:     {tyre_raw.shape}')

In [ ]:
# Sample raw race data
if not race_raw.empty:
    print('First 10 rows of race_data_raw.csv:')
    display(race_raw.head(10))

In [ ]:
# Sample pit stop data
if not pitstop_raw.empty:
    print('First 10 rows of pitstop_data_raw.csv:')
    display(pitstop_raw.head(10))

In [ ]:
# Collection summary
if not race_raw.empty:
    print('Data Collection Summary')
    print('=' * 45)
    for yr in [2022, 2023, 2024]:
        sub = race_raw[race_raw['Year'] == yr]
        print(f'  {yr}: {len(sub):4d} entries, {sub["Race"].nunique():2d} races, {sub["Driver"].nunique():2d} drivers')
    print(f'\n  Total: {len(race_raw)} race entries')
    print(f'  Total: {len(pitstop_raw)} pit stop records')

In [ ]:
# Bar chart: records per race per season
if not race_raw.empty:
    counts = race_raw.groupby(['Year','Race']).size().reset_index(name='Count')
    pivot  = counts.pivot(index='Race', columns='Year', values='Count').fillna(0)
    
    fig, ax = plt.subplots(figsize=(16, 6))
    x = np.arange(len(pivot))
    w = 0.25
    for i, yr in enumerate(sorted(pivot.columns)):
        ax.bar(x + i*w, pivot[yr], w, label=str(int(yr)),
               color=[F1_RED, F1_ORANGE, '#27F4D2'][i], edgecolor='white', lw=0.3)
    ax.set_xticks(x + w); ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Records'); ax.set_title('Records Collected Per Race Per Season', fontsize=14, fontweight='bold')
    ax.legend(title='Season')
    plt.tight_layout(); plt.show()

---
## 3. Data Preprocessing

The preprocessing pipeline (`src/preprocessing.py`) performs:
- **Cleaning**: drops nulls, removes pit stops < 1.5s or > 60s
- **Feature Engineering**: IsWinner, AvgPitStops, TyreAge, StartingCompound
- **Encoding**: LabelEncoder for Team, Driver, Circuit, Compound
- **Split**: 80% train / 20% test with random_state=42
- **Scaling**: StandardScaler for zero-mean unit-variance features


In [ ]:
race_p1      = load_csv('race_winner_data.csv', PROCESSED_DIR)
pitstop_p2   = load_csv('pitstop_duration_data.csv', PROCESSED_DIR)
race_full    = load_csv('race_full_processed.csv', PROCESSED_DIR)
pitstop_full = load_csv('pitstop_full_processed.csv', PROCESSED_DIR)

print(f'Problem 1 dataset shape: {race_p1.shape}')
print(f'Problem 2 dataset shape: {pitstop_p2.shape}')

In [ ]:
if not race_p1.empty:
    print('Problem 1 feature columns:', list(race_p1.columns))
    display(race_p1.describe())

In [ ]:
if not pitstop_p2.empty:
    print('Problem 2 feature columns:', list(pitstop_p2.columns))
    display(pitstop_p2.describe())

In [ ]:
# Missing values heatmap (after cleaning)
if not race_p1.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    missing = race_p1.isnull().astype(int)
    if missing.sum().sum() == 0:
        ax.text(0.5, 0.5, 'No Missing Values after Cleaning!',
                ha='center', va='center', fontsize=20, color='#27F4D2', fontweight='bold',
                transform=ax.transAxes)
    else:
        sns.heatmap(missing, cbar=True, cmap=['#16213e', F1_RED], yticklabels=False, ax=ax)
    ax.set_title('Missing Values — After Cleaning', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Train/test split pie
if not race_p1.empty:
    n_total = len(race_p1)
    n_train = int(n_total * 0.8); n_test = n_total - n_train
    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        [n_train, n_test],
        labels=[f'Train ({n_train})', f'Test ({n_test})'],
        colors=[F1_RED, F1_ORANGE], autopct='%1.1f%%', startangle=90
    )
    for t in autotexts: t.set_fontweight('bold'); t.set_fontsize(13)
    ax.set_title('80/20 Train/Test Split', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 4. Exploratory Data Analysis

Before modelling, we explore the data to understand team performance, circuit difficulty, tyre usage, and the relationship between grid position and finishing position.


In [ ]:
# Team performance
if not race_full.empty:
    team_pts = race_full.groupby('Team')['Points'].sum().sort_values()
    colors   = [TEAM_COLORS.get(t, '#AAA') for t in team_pts.index]
    fig, ax  = plt.subplots(figsize=(12, 7))
    ax.barh(range(len(team_pts)), team_pts.values, color=colors, edgecolor='white', lw=0.5)
    ax.set_yticks(range(len(team_pts))); ax.set_yticklabels(team_pts.index)
    ax.set_xlabel('Total Points 2022-2024')
    ax.set_title('Constructor Championship Points', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Grid vs finish scatter
if not race_full.empty and 'GridPosition' in race_full.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    for team in race_full['Team'].unique():
        m = race_full['Team'] == team
        ax.scatter(race_full.loc[m,'GridPosition'], race_full.loc[m,'Position'],
                   c=TEAM_COLORS.get(team,'#AAA'), s=25, alpha=0.5, label=team, edgecolors='white', lw=0.2)
    ax.plot([0,22],[0,22],'--',color=F1_RED,lw=2,label='Grid = Finish')
    ax.set_xlabel('Grid Position'); ax.set_ylabel('Finishing Position')
    ax.set_title('Grid vs Finishing Position', fontsize=14, fontweight='bold')
    ax.legend(fontsize=7, ncol=2); ax.invert_yaxis()
    plt.tight_layout(); plt.show()

In [ ]:
# Feature correlation heatmap
if not race_p1.empty:
    corr = race_p1.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0.5, ax=ax)
    ax.set_title('Feature Correlation — Race Winner', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Pit stop violin by team
if not pitstop_full.empty and 'Team' in pitstop_full.columns:
    order = pitstop_full.groupby('Team')['PitStopDuration'].median().sort_values().index
    palette = {t: TEAM_COLORS.get(t,'#AAA') for t in pitstop_full['Team'].unique()}
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.violinplot(data=pitstop_full, x='Team', y='PitStopDuration',
                   order=order, palette=palette, inner='box', ax=ax)
    ax.set_xlabel('Team'); ax.set_ylabel('Pit Stop Duration (s)')
    ax.set_title('Pit Stop Duration by Team', fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

In [ ]:
# Tyre compound usage pie per season
if not tyre_raw.empty and 'Compound' in tyre_raw.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for idx, yr in enumerate([2022, 2023, 2024]):
        ax = axes[idx]
        sub = tyre_raw[tyre_raw['Year'] == yr] if 'Year' in tyre_raw.columns else tyre_raw
        counts = sub['Compound'].value_counts()
        colors_t = [TYRE_COLORS.get(c,'#AAA') for c in counts.index]
        ax.pie(counts.values, labels=counts.index, colors=colors_t,
               autopct='%1.1f%%', startangle=90, textprops={'color':'white'})
        ax.set_title(f'Season {yr}', fontsize=13, fontweight='bold')
    fig.suptitle('Tyre Compound Usage Per Season', fontsize=16, fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 5. Problem 1: Race Winner Prediction

We predict the finishing position of each driver.

**Target:** `Position` (1 = winner)  
**Features:** GridPosition, TeamEncoded, DriverEncoded, CircuitEncoded, AirTemp, TrackTemp, Rainfall, AvgPitStops, StartingCompoundEncoded

**Key insight:** Grid position is the single strongest predictor — drivers who qualify well tend to finish well.


In [ ]:
# Prepare Problem 1 data
if not race_p1.empty:
    TARGET = 'Position'
    FEATURES = [c for c in race_p1.columns if c != TARGET]
    X = race_p1[FEATURES]; y = race_p1[TARGET]

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    sc = StandardScaler()
    Xtr_sc = pd.DataFrame(sc.fit_transform(X_tr), columns=FEATURES, index=X_tr.index)
    Xte_sc = pd.DataFrame(sc.transform(X_te),     columns=FEATURES, index=X_te.index)
    print(f'Train: {Xtr_sc.shape}  |  Test: {Xte_sc.shape}')
    print(f'Features: {FEATURES}')

In [ ]:
# Train Linear Regression
if not race_p1.empty:
    lr = LinearRegression()
    lr.fit(Xtr_sc, y_tr)
    yp_lr = lr.predict(Xte_sc)

    lr_mse  = mean_squared_error(y_te, yp_lr)
    lr_rmse = np.sqrt(lr_mse)
    lr_mae  = mean_absolute_error(y_te, yp_lr)
    lr_r2   = r2_score(y_te, yp_lr)
    lr_cv   = np.sqrt(-cross_val_score(lr, Xtr_sc, y_tr, cv=5, scoring='neg_mean_squared_error'))

    print('Linear Regression — Race Winner Prediction')
    print(f'  MSE:           {lr_mse:.4f}')
    print(f'  RMSE:          {lr_rmse:.4f}')
    print(f'  MAE:           {lr_mae:.4f}')
    print(f'  R2:            {lr_r2:.4f}')
    print(f'  CV RMSE:       {lr_cv.mean():.4f} +/- {lr_cv.std():.4f}')

In [ ]:
# KNN Classifier — find best K
if not race_p1.empty:
    y_tr_c = y_tr.astype(int); y_te_c = y_te.astype(int)
    k_vals = list(range(1, 21))
    cv_acc = [cross_val_score(KNeighborsClassifier(n_neighbors=k), Xtr_sc, y_tr_c, cv=5, scoring='accuracy').mean()
              for k in k_vals]
    best_k = k_vals[np.argmax(cv_acc)]
    print(f'Best K = {best_k}  (CV Accuracy = {max(cv_acc):.4f})')

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(k_vals, cv_acc, '-o', color=F1_RED, lw=2, ms=8)
    ax.axvline(x=best_k, color=F1_ORANGE, ls='--', label=f'Best K={best_k}')
    ax.set_xlabel('K'); ax.set_ylabel('CV Accuracy')
    ax.set_title('KNN — Optimal K Selection', fontsize=14, fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Train final KNN
if not race_p1.empty:
    knn = KNeighborsClassifier(n_neighbors=best_k)
    knn.fit(Xtr_sc, y_tr_c)
    yp_knn = knn.predict(Xte_sc)

    knn_acc  = accuracy_score(y_te_c, yp_knn)
    knn_prec = precision_score(y_te_c, yp_knn, average='weighted', zero_division=0)
    knn_rec  = recall_score(y_te_c, yp_knn, average='weighted', zero_division=0)
    knn_f1   = f1_score(y_te_c, yp_knn, average='weighted', zero_division=0)
    knn_mse  = mean_squared_error(y_te_c, yp_knn)
    knn_rmse = np.sqrt(knn_mse)

    print(f'KNN Classifier (K={best_k})')
    print(f'  Accuracy:  {knn_acc:.4f}')
    print(f'  Precision: {knn_prec:.4f}')
    print(f'  Recall:    {knn_rec:.4f}')
    print(f'  F1 Score:  {knn_f1:.4f}')
    print(f'  RMSE:      {knn_rmse:.4f}')

In [ ]:
# LR actual vs predicted
if not race_p1.empty:
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(y_te, yp_lr, c=F1_ORANGE, s=40, alpha=0.6, edgecolors='white', lw=0.3)
    mn, mx = min(y_te.min(), yp_lr.min()), max(y_te.max(), yp_lr.max())
    ax.plot([mn, mx], [mn, mx], '--', color=F1_RED, lw=2, label='Perfect Prediction')
    z = np.polyfit(y_te, yp_lr, 1); p = np.poly1d(z)
    xln = np.linspace(mn, mx, 100)
    ax.plot(xln, p(xln), color='#27F4D2', lw=2, label='Regression Line')
    ax.set_xlabel('Actual Position'); ax.set_ylabel('Predicted Position')
    ax.set_title('LR Actual vs Predicted — Race Position', fontsize=14, fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# KNN confusion matrix
if not race_p1.empty:
    labels = sorted(set(y_te_c)|set(yp_knn))[:10]
    cm = confusion_matrix(y_te_c, yp_knn, labels=labels)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('KNN Confusion Matrix (Top 10 Positions)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 6. Problem 2: Pit Stop Duration Prediction

We predict how long each pit stop takes in seconds.

**Target:** `PitStopDuration` (seconds)  
**Features:** TeamEncoded, TyreCompoundEncoded, LapNumber, CircuitEncoded, AirTemp, TrackTemp, TyreAge, RaceYear, DriverEncoded

**Key insight:** Team pit crew capability and tyre compound have the biggest impact on duration.


In [ ]:
# Prepare Problem 2 data
if not pitstop_p2.empty:
    TARGET2 = 'PitStopDuration'
    FEAT2   = [c for c in pitstop_p2.columns if c != TARGET2]
    X2 = pitstop_p2[FEAT2]; y2 = pitstop_p2[TARGET2]

    X2tr, X2te, y2tr, y2te = train_test_split(X2, y2, test_size=0.2, random_state=42)
    sc2 = StandardScaler()
    X2tr_sc = pd.DataFrame(sc2.fit_transform(X2tr), columns=FEAT2, index=X2tr.index)
    X2te_sc = pd.DataFrame(sc2.transform(X2te),     columns=FEAT2, index=X2te.index)
    print(f'Train: {X2tr_sc.shape}  |  Test: {X2te_sc.shape}')
    print(f'Mean duration: {y2.mean():.2f}s  |  Std: {y2.std():.2f}s')

In [ ]:
# Linear Regression — Pit Stops
if not pitstop_p2.empty:
    lr2 = LinearRegression()
    lr2.fit(X2tr_sc, y2tr)
    yp2_lr = lr2.predict(X2te_sc)

    lr2_mse  = mean_squared_error(y2te, yp2_lr)
    lr2_rmse = np.sqrt(lr2_mse)
    lr2_mae  = mean_absolute_error(y2te, yp2_lr)
    lr2_r2   = r2_score(y2te, yp2_lr)
    lr2_cv   = np.sqrt(-cross_val_score(lr2, X2tr_sc, y2tr, cv=5, scoring='neg_mean_squared_error'))

    print('Linear Regression — Pit Stop Duration')
    print(f'  RMSE: {lr2_rmse:.4f}s  |  MAE: {lr2_mae:.4f}s  |  R2: {lr2_r2:.4f}')
    print(f'  CV RMSE: {lr2_cv.mean():.4f} +/- {lr2_cv.std():.4f}')

In [ ]:
# KNN Regressor — find best K
if not pitstop_p2.empty:
    k2_vals  = list(range(1, 21))
    cv_rmse2 = [np.sqrt(-cross_val_score(KNeighborsRegressor(n_neighbors=k), X2tr_sc, y2tr,
                                          cv=5, scoring='neg_mean_squared_error')).mean()
                for k in k2_vals]
    best_k2 = k2_vals[np.argmin(cv_rmse2)]
    print(f'Best K = {best_k2}  (CV RMSE = {min(cv_rmse2):.4f}s)')

    knn2 = KNeighborsRegressor(n_neighbors=best_k2)
    knn2.fit(X2tr_sc, y2tr)
    yp2_knn = knn2.predict(X2te_sc)

    knn2_mse  = mean_squared_error(y2te, yp2_knn)
    knn2_rmse = np.sqrt(knn2_mse)
    knn2_mae  = mean_absolute_error(y2te, yp2_knn)
    knn2_r2   = r2_score(y2te, yp2_knn)

    print(f'KNN Regressor (K={best_k2})')
    print(f'  RMSE: {knn2_rmse:.4f}s  |  MAE: {knn2_mae:.4f}s  |  R2: {knn2_r2:.4f}')

In [ ]:
# Actual vs predicted for both models
if not pitstop_p2.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    for ax, yp, color, label in [(ax1, yp2_lr, F1_ORANGE, 'Linear Regression'),
                                   (ax2, yp2_knn, '#27F4D2', f'KNN (K={best_k2})')]:
        ax.scatter(y2te, yp, c=color, s=20, alpha=0.5, edgecolors='white', lw=0.2)
        mn, mx = min(y2te.min(), yp.min()), max(y2te.max(), yp.max())
        ax.plot([mn, mx], [mn, mx], '--', color=F1_RED, lw=2)
        ax.set_xlabel('Actual Duration (s)'); ax.set_ylabel('Predicted Duration (s)')
        ax.set_title(label, fontsize=13, fontweight='bold')
    fig.suptitle('Pit Stop Duration — Actual vs Predicted', fontsize=15, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Residual plots side by side
if not pitstop_p2.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    for ax, yp, res_color, label in [(ax1, yp2_lr, F1_RED, 'LR Residuals'),
                                      (ax2, yp2_knn, '#27F4D2', 'KNN Residuals')]:
        res = y2te.values - yp
        ax.scatter(yp, res, c=res_color, s=20, alpha=0.5)
        ax.axhline(y=0, color=F1_ORANGE, ls='--', lw=2)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
        ax.set_title(label, fontsize=13, fontweight='bold')
    fig.suptitle('Residual Analysis — Pit Stop Duration', fontsize=15, fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 7. Final Model Comparison

Here we compare all four models — LR and KNN for both problems — using a comprehensive metrics table.


In [ ]:
# Full comparison table
if not race_p1.empty and not pitstop_p2.empty:
    rows = [
        ['RMSE',        f'{lr_rmse:.4f}',  f'{knn_rmse:.4f}',   f'{lr2_rmse:.4f}',  f'{knn2_rmse:.4f}'],
        ['MAE',         f'{lr_mae:.4f}',   f'{knn_rmse:.4f}',   f'{lr2_mae:.4f}',   f'{knn2_mae:.4f}'],
        ['R2',          f'{lr_r2:.4f}',    'N/A',                f'{lr2_r2:.4f}',    f'{knn2_r2:.4f}'],
        ['Accuracy',    'N/A',             f'{knn_acc:.4f}',     'N/A',              'N/A'],
        ['F1 Score',    'N/A',             f'{knn_f1:.4f}',      'N/A',              'N/A'],
    ]
    comparison = pd.DataFrame(rows,
        columns=['Metric', 'P1: LR', f'P1: KNN(K={best_k})', 'P2: LR', f'P2: KNN(K={best_k2})'])
    print('FULL ALGORITHM COMPARISON TABLE')
    print('=' * 75)
    display(comparison)
    
    # Verdict
    print()
    if lr_rmse < knn_rmse:
        print(f'Problem 1 Winner: Linear Regression (RMSE {lr_rmse:.4f} vs {knn_rmse:.4f})')
    else:
        print(f'Problem 1 Winner: KNN (RMSE {knn_rmse:.4f} vs {lr_rmse:.4f})')
    if lr2_rmse < knn2_rmse:
        print(f'Problem 2 Winner: Linear Regression (RMSE {lr2_rmse:.4f}s vs {knn2_rmse:.4f}s)')
    else:
        print(f'Problem 2 Winner: KNN (RMSE {knn2_rmse:.4f}s vs {lr2_rmse:.4f}s)')

In [ ]:
# Feature importance (LR coefficients)
if not race_p1.empty:
    fi = np.abs(lr.coef_)
    fi_idx = np.argsort(fi)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh([FEATURES[i] for i in fi_idx], fi[fi_idx],
            color=plt.cm.Reds(np.linspace(0.3, 1.0, len(fi))), edgecolor='white', lw=0.5)
    ax.set_xlabel('Coefficient Magnitude')
    ax.set_title('Feature Importance (LR Coefficients)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Learning curves for Problem 1 LR
if not race_p1.empty:
    train_sz, tr_sc, te_sc = learning_curve(
        LinearRegression(), Xtr_sc, y_tr,
        train_sizes=np.linspace(0.1, 1.0, 10), cv=5,
        scoring='neg_mean_squared_error', n_jobs=-1
    )
    tr_rmse = np.sqrt(-tr_sc.mean(axis=1))
    te_rmse = np.sqrt(-te_sc.mean(axis=1))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(train_sz, tr_rmse, '-o', color=F1_RED,    lw=2, label='Train RMSE')
    ax.plot(train_sz, te_rmse, '--s', color=F1_ORANGE, lw=2, label='CV RMSE')
    ax.set_xlabel('Training Set Size'); ax.set_ylabel('RMSE')
    ax.set_title('Learning Curve — LR Race Winner Prediction', fontsize=14, fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Display saved visualizations
from IPython.display import Image, display as ipydisplay
PLOTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'plots')

print('Saved F1-themed visualizations:')
for f in sorted(os.listdir(PLOTS_DIR)):
    if f.endswith('.png'):
        print(f'  {f}')

---
## 8. Conclusion

### Key Findings

**Problem 1 — Race Winner Prediction:**
- Linear Regression outperforms KNN (lower RMSE)
- Grid position is the strongest single predictor of finishing position
- Weather and tyre compound add marginal but meaningful predictive value

**Problem 2 — Pit Stop Duration:**
- Linear Regression achieves slightly better RMSE than KNN
- Team capability (encoded as TeamEncoded) is the dominant feature
- Tyre compound type has a notable effect on pit stop duration

**Algorithm Comparison:**
- Linear Regression wins both problems on RMSE
- KNN is useful when non-linear patterns exist, but needs more data
- Both algorithms benefit significantly from feature scaling

### Future Work

- Apply ensemble methods (Random Forest, Gradient Boosting)
- Incorporate time series patterns and momentum
- Build a real-time prediction dashboard
- Add tyre degradation models and fuel load data


In [ ]:
print('LIGHTS OUT AND AWAY — Project Complete!')
print()
print('All visualizations: outputs/plots/')
print('PDF Report:         outputs/reports/lights_out_and_away_report.pdf')
print('Trained models:     models/')
print()
print('Run `python main.py` to regenerate everything from scratch.')